# Vapour Compression and Brayton HPR

**Learning outcome:** Apply vapour compression and brayton hpr through the public `PinchProblem` or `PinchWorkspace` workflow.

**Level:** Advanced  
**Execution profile:** `slow-hpr`  
**Expected runtime:** 5 to 30 minutes  
**Optional extras:** hpr

The lifecycle is explicit: prepare the study, run the named method, then inspect cached results. Observation cells do not launch analysis.

## Study question and data

**Study question:** Which simulated heat-pump family and working fluid best fit the required temperature lift?

The sample data is packaged with OpenPinch, so the notebook runs without path setup. Read the named inputs and assumptions before substituting plant data.

## Step 1: Evaluate vapour-compression candidates

Run this cell once, then inspect its named outputs. Arguments on the method call apply to this analysis; stored configuration is only the fallback when an argument is omitted.

In [ ]:
from OpenPinch import PinchProblem

problem = PinchProblem("heat_pump_targeting.json", project_name="HPR Models")
def screen_cycle(method, **arguments):
    try:
        return {"status": "feasible", "result": method(**arguments)}
    except (ValueError, NotImplementedError) as error:
        return {"status": "no feasible solution", "reason": str(error)}

vapour_compression = screen_cycle(
    problem.target.vapour_compression_heat_pump,
    refrigerants=["water", "ammonia"],
    load_fraction=0.25,
    maximum_restarts=1,
)
vapour_compression

## Step 2: Compare refrigeration and Brayton variants

Run this cell once, then inspect its named outputs. Arguments on the method call apply to this analysis; stored configuration is only the fallback when an argument is omitted.

In [ ]:
vc_refrigeration = screen_cycle(
    problem.target.vapour_compression_refrigeration,
    refrigerants=["ammonia"],
    load_fraction=0.25,
    maximum_restarts=1,
)
brayton = screen_cycle(
    problem.target.brayton_heat_pump,
    load_fraction=0.25, maximum_restarts=1
)
brayton_refrigeration = screen_cycle(
    problem.target.brayton_refrigeration,
    load_fraction=0.25, maximum_restarts=1
)

## Review the result

Compare feasibility and returned cycle results on the same process basis before selecting a thermodynamic model.

In [ ]:
from IPython.display import display

cycle_comparison = {
    "vapour compression heat pump": vapour_compression,
    "vapour compression refrigeration": vc_refrigeration,
    "Brayton heat pump": brayton,
    "Brayton refrigeration": brayton_refrigeration,
}
display(cycle_comparison)

## Interpret the result

Reject candidates on feasibility before ranking energy performance; Brayton and vapour-compression results are not interchangeable design models.

## Adapt this template

Use a short, defensible refrigerant list and record why each cycle family is in scope.

Keep the workflow explicit: prepare input, call one named engineering method, inspect cached results, then export.